# Model Tester

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

## Setup

In [ ]:
import sys
import time
import logging

import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, LearningRateSchedule

sys.path.append(os.path.join(os.getcwd(), ".."))
from scripts.models import ModelCore, AutoModel
from scripts.models.modelcore import get_model_class
from scripts.utils import setup_logging, load_data, get_fisher, try_init_wandb
from scripts.utils.plots import plot_predictions, plot_histogram
from scripts.utils.tf.plots import plot_metrics
from scripts.utils.tf.callbacks import TimedLoggingCallback, WarmupLearningRate

In [ ]:
logger = setup_logging(__name__, level=logging.INFO)
logging.getLogger("scripts").setLevel(logging.DEBUG)
logging.getLogger("tensorflow").setLevel(logging.ERROR)

## Configure

In [ ]:
args = ["settings/planck.json", "--nsims", "100"]
# core = Core(["settings/planck.json", "--nsims", "100"], trainer=True)
# core = Core(["settings/heidelberg.json", "--nsims", "100"], trainer=True)

MAX_EPOCHS = 100
BATCH_SIZE = 16

# just some info for the model name
timestamp = int(time.time())
model_settings = {
    "name": f"tester_{timestamp}",
}

data_settings = {
    "shuffle": False,
    "shuffle_buffer": 1000,
    "seed": None,
    "batch_size": BATCH_SIZE,
    "cache": True,
    "normalize": False,
}

# additional metrics we are interested in
metrics = ["mean_absolute_error"]

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    TimedLoggingCallback(print_frequency=3),
    # TensorBoard(log_dir=f"{s.tb_dir}/{model_settings['name']}"),
    TerminateOnNaN(),
]

In [ ]:
# try_init_wandb(notes="model testing", tags=["notebook"], append_to=callbacks)

## Isensee Model

In [ ]:
model = AutoModel(args + ["--model", "ISENSEE"])
# model = get_model_class("ISENSEE")(args)
train_ds, test_ds, val_ds = model.init_dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(model.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## ALM Model

In [ ]:
model = get_model_class("ALM")(args)
train_ds, test_ds, val_ds = model.init_dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-5, 1000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(model.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## DCNN Model

In [ ]:
model = get_model_class("dcnn")(args)
train_ds, test_ds, val_ds = model.init_dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(model.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## RESNET

In [ ]:
model = get_model_class("RESNET")(args)
train_ds, test_ds, val_ds = model.init_dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(model.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## NAGARAJAPPA

In [ ]:
model = get_model_class("NAGARAJAPPA")(args)
train_ds, test_ds, val_ds = model.init_dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(model.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)